# MaveDB quickstart

Download and standardize one fixed MaveDB score set through the public API. Source metadata can supply a usable protein reference, so this workflow does not guess or reconstruct one manually.

In [ ]:
from pathlib import Path

from dmsroute import (
    FilesystemCache,
    download_and_standardize_dataset,
    get_dataset_metadata,
    read_table,
)

In [ ]:
DATASET_ID = "urn:mavedb:00000001-a-4"
CACHE_DIR = Path("datasets/.cache")
OUTPUT_DIR = Path("datasets/mavedb/ube2i")

cache = FilesystemCache(CACHE_DIR)

## Inspect normalized metadata

In [ ]:
metadata = get_dataset_metadata("mavedb", DATASET_ID)

{
    "source": metadata.source,
    "dataset_id": metadata.dataset_id,
    "title": metadata.title,
    "target_id": metadata.target_id,
    "variant_type": metadata.variant_type,
    "n_variants": metadata.n_variants,
}

## Download and standardize

The download helper resolves the score table, metadata, reference sequence, and standard output bundle. Optional score transformations remain disabled.

In [ ]:
result = download_and_standardize_dataset(
    "mavedb",
    DATASET_ID,
    output_dir=OUTPUT_DIR,
    cache=cache,
    overwrite=True,
)

## Inspect the standardized result

In [ ]:
dataset = read_table(result.dataset_path)
preview_columns = [
    column
    for column in (
        "hgvs_pro",
        "variant",
        "is_wildtype",
        "n_mutations",
        "score_raw",
        "status",
        "error",
    )
    if column in dataset.columns
]
dataset.loc[:, preview_columns].head()

In [ ]:
dataset["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows")

In [ ]:
{
    "dataset_path": result.dataset_path,
    "summary_csv_path": result.summary_csv_path,
    "summary_json_path": result.summary_json_path,
    "summary": {
        key: result.summary.get(key)
        for key in (
            "status",
            "raw_rows",
            "output_rows",
            "validated_rows",
            "wt_sequence_provenance",
        )
    },
}